# Clean `single-player-games` for ML (price vs sales proxy)

This notebook turns [`data/single-player-games.csv`](data/single-player-games.csv) into a numeric dataset for modeling.

**Sales proxy:** `steamspy_owners` is parsed from SteamSpy’s **estimated owner range** (e.g. `200,000 .. 500,000`) into `owners_low`, `owners_high`, `owners_mid`, and `log_owners_mid`. These are **not** true unit sales; any model measures association with this proxy, not causal “optimal price.”

**Outputs:** `data/single-player-games-cleaned.parquet` (primary) and optional `data/single-player-games-cleaned.csv`.

## 1. Configuration

Adjust paths and knobs here before running the rest.

In [ ]:
from pathlib import Path

SCRIPT_DIR = Path.cwd()
INPUT_CSV = SCRIPT_DIR / "data" / "single-player-games.csv"
OUTPUT_PARQUET = SCRIPT_DIR / "data" / "single-player-games-cleaned.parquet"
OUTPUT_CSV = SCRIPT_DIR / "data" / "single-player-games-cleaned.csv"
WRITE_CSV_MIRROR = False  # set True for spreadsheet mirror (not committed; see .gitignore)

# None = today UTC midnight; or set e.g. "2026-05-05" for reproducible age_days
REFERENCE_DATE = None

GENRE_TOP_K = 20
TAG_TOP_K = 15
MIN_OWNERS_MID = None  # e.g. 5000.0 to drop very small estimates


## 2. Imports and helpers

The next cell installs `numpy`, `pandas`, and `pyarrow` into the **active kernel** if they are missing (fixes `ModuleNotFoundError` when the kernel is not your project `.venv`).

In [ ]:
import importlib.util
import json
import re
import subprocess
import sys
from collections import Counter

_missing = ("numpy", "pandas", "pyarrow", "matplotlib", "seaborn")
if any(importlib.util.find_spec(p) is None for p in _missing):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

OWNERS_PATTERN = re.compile(r"([\d,]+)\s*\.\.\s*([\d,]+)")


def _strip_commas_num(s: str) -> int:
    return int(s.replace(",", "").strip())


def parse_owners_range(raw: object) -> tuple[float | None, float | None]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None, None
    text = str(raw).strip()
    if not text:
        return None, None
    m = OWNERS_PATTERN.search(text)
    if not m:
        return None, None
    try:
        low = _strip_commas_num(m.group(1))
        high = _strip_commas_num(m.group(2))
    except ValueError:
        return None, None
    if high < low:
        low, high = high, low
    return float(low), float(high)


def split_semicolon_counts(series: pd.Series) -> pd.Series:
    def count_parts(x: object) -> int:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 0
        parts = [p.strip() for p in str(x).split(";") if p.strip()]
        return len(parts)

    return series.map(count_parts)


def split_genre_list(raw: object) -> list[str]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    return [g.strip() for g in str(raw).split(",") if g.strip()]


def sanitize_feature_name(s: str) -> str:
    out = re.sub(r"[^\w]+", "_", s.strip())
    out = re.sub(r"_+", "_", out).strip("_")
    return out or "unknown"


def parse_tags_dict(raw: object) -> dict:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return {}
    text = str(raw).strip()
    if not text:
        return {}
    try:
        data = json.loads(text)
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}


## 3. Load raw CSV

In [ ]:
df = pd.read_csv(INPUT_CSV, dtype={"appid": "Int64"})
n_read = len(df)
print(f"Loaded {n_read} rows from {INPUT_CSV}")
df.head(2)


## 4. Parse features (owners, dates, price, tags, genres)

In [ ]:
# Naive datetimes only (ISO dates have no tz). Avoid tz-aware utcnow() vs naive release_dt.
ref = (
    pd.Timestamp(REFERENCE_DATE).normalize()
    if REFERENCE_DATE
    else pd.Timestamp(pd.Timestamp.now(tz="UTC").date())
)

low_list: list[float | None] = []
high_list: list[float | None] = []
for v in df["steamspy_owners"]:
    lo, hi = parse_owners_range(v)
    low_list.append(lo)
    high_list.append(hi)

df["owners_low"] = low_list
df["owners_high"] = high_list
df["owners_mid"] = (df["owners_low"] + df["owners_high"]) / 2.0
df["log_owners_mid"] = np.where(
    df["owners_mid"].notna() & (df["owners_mid"] > 0),
    np.log(df["owners_mid"]),
    np.nan,
)

df["release_dt"] = pd.to_datetime(df["release_date_iso"], errors="coerce")
df["release_year"] = df["release_dt"].dt.year
df["age_days"] = (ref - df["release_dt"]).dt.days

df["price_usd"] = df["price_final_cents"].astype("float64") / 100.0

df["developer_count"] = split_semicolon_counts(df["developers"])
df["publisher_count"] = split_semicolon_counts(df["publishers"])

tags_parsed = df["steamspy_tags"].map(parse_tags_dict)
df["tag_count"] = tags_parsed.map(len)
df["has_tags"] = df["tag_count"] > 0

df["genre_list"] = df["steamspy_genre"].map(split_genre_list)
df["primary_genre"] = df["genre_list"].map(lambda xs: xs[0] if xs else np.nan)


## 5. Filter rows (sequential drops)

In [ ]:
mask_owners = df["owners_low"].notna() & df["owners_high"].notna()
n_bad_owners = int((~mask_owners).sum())
df = df.loc[mask_owners].copy()

mask_price = df["price_final_cents"].notna()
n_bad_price = int((~mask_price).sum())
df = df.loc[mask_price].copy()

mask_date = df["release_dt"].notna()
n_bad_date = int((~mask_date).sum())
df = df.loc[mask_date].copy()

if MIN_OWNERS_MID is not None:
    mask_min = df["owners_mid"] >= MIN_OWNERS_MID
    n_min_owners = int((~mask_min).sum())
    df = df.loc[mask_min].copy()
else:
    n_min_owners = 0

print(f"Reference date (age_days): {ref.date()}")
print(f"Dropped (unparseable owners): {n_bad_owners}")
print(f"Dropped (missing price): {n_bad_price}")
print(f"Dropped (invalid release date): {n_bad_date}")
if MIN_OWNERS_MID is not None:
    print(f"Dropped (owners_mid < {MIN_OWNERS_MID}): {n_min_owners}")
print(f"Remaining rows: {len(df)}")


## 6. Top-K genre multi-hot columns

In [ ]:
k = max(0, GENRE_TOP_K)
genre_counter: Counter[str] = Counter()
for lst in df["genre_list"]:
    for g in lst:
        genre_counter[g] += 1
top_genres = [g for g, _ in genre_counter.most_common(k)]

used_names: dict[str, str] = {}
genre_cols: list[str] = []
for g in top_genres:
    base = sanitize_feature_name(g)
    name = base
    i = 2
    while name in used_names.values():
        name = f"{base}_{i}"
        i += 1
    used_names[g] = name
    col = f"genre__{name}"
    genre_cols.append(col)
    df[col] = df["genre_list"].map(lambda lst, gg=g: int(gg in lst))

df = df.drop(columns=["genre_list"])
print(f"Genre multi-hot columns: {len(genre_cols)}")


## 6b. Top-K Steam tag multi-hot columns

In [ ]:
k_tag = max(0, TAG_TOP_K)
tag_counter: Counter[str] = Counter()
for raw in df["steamspy_tags"]:
    for tag in parse_tags_dict(raw):
        tag_counter[tag] += 1
top_tags = [t for t, _ in tag_counter.most_common(k_tag)]

used_tag_names: dict[str, str] = {}
tag_cols: list[str] = []
for t in top_tags:
    base = sanitize_feature_name(t)
    name = base
    i = 2
    while name in used_tag_names.values():
        name = f"{base}_{i}"
        i += 1
    used_tag_names[t] = name
    col = f"tag__{name}"
    tag_cols.append(col)
    df[col] = df["steamspy_tags"].map(
        lambda raw, tt=t: int(tt in parse_tags_dict(raw))
    )

df_clean = df
print(f"Tag multi-hot columns: {len(tag_cols)}")

## 7. Validate, save, quick peek

**Column glossary (added / key fields):**

| Column | Meaning |
|--------|---------|
| `owners_mid`, `log_owners_mid` | Midpoint of SteamSpy owner range; log for skewed targets |
| `price_usd`, `price_discount_percent` | Store price and discount |
| `release_dt`, `release_year`, `age_days` | Parsed release date and age vs reference |
| `steamspy_ccu`, `steamspy_median_*` | Engagement (many zeros are normal) |
| `developer_count`, `publisher_count` | Count of `;`-separated names |
| `tag_count`, `has_tags` | Parsed JSON tag dict size |
| `primary_genre` | First genre in SteamSpy genre string |
| `genre__*` | Multi-hot for top-K genres |
| `tag__*` | Multi-hot for top-K SteamSpy tags |


In [ ]:
required = [
    "appid",
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "release_dt",
    "age_days",
]
missing = [c for c in required if c not in df_clean.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Wrote Parquet: {OUTPUT_PARQUET} ({len(df_clean)} rows)")

if WRITE_CSV_MIRROR:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote CSV: {OUTPUT_CSV}")

df_clean[["name", "owners_mid", "price_usd", "age_days", "primary_genre"]].head()


In [ ]:
df_clean[["owners_mid", "price_usd", "age_days", "steamspy_ccu"]].describe()


## 8. Visualizations (saved to `visualizations/`)

These are common EDA plots you’ll typically generate before training ML models (distribution checks, target skew, price vs sales proxy, correlations, and category effects).

In [ ]:
from pathlib import Path

VIS_DIR = SCRIPT_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving plots to: {VIS_DIR}")

# If you open this notebook without re-running earlier cells, reload the cleaned dataset.
if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

# Convenience columns
_df = df_clean.copy()
_df["log_price_usd"] = np.where(_df["price_usd"] > 0, np.log(_df["price_usd"]), np.nan)

# Robust y for plotting (avoid inf)
_df["log_owners_mid"] = np.where(_df["owners_mid"] > 0, np.log(_df["owners_mid"]), np.nan)

# Limit extreme owners for clearer scatter (keep full data for modeling)
owners_cap = _df["owners_mid"].quantile(0.995)
_df_scatter = _df[_df["owners_mid"] <= owners_cap].copy()
print(f"Scatter cap owners_mid at p99.5={owners_cap:,.0f}; rows kept={len(_df_scatter)}")


In [ ]:
# 1) Price distribution
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["price_usd"], bins=60, kde=False, ax=ax)
ax.set_title("Price distribution (USD)")
ax.set_xlabel("price_usd")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_usd_hist.png", dpi=200)
plt.close(fig)

# 2) Owners distribution (log scale)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["log_owners_mid"].dropna(), bins=60, kde=False, ax=ax)
ax.set_title("SteamSpy owners_mid distribution (log)")
ax.set_xlabel("log(owners_mid)")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(VIS_DIR / "owners_mid_log_hist.png", dpi=200)
plt.close(fig)

# 3) Price vs owners (hexbin-like view using seaborn scatter with alpha)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_usd"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("Price vs log(owners_mid) (capped at p99.5 owners)")
ax.set_xlabel("price_usd")
ax.set_ylabel("log(owners_mid)")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

# 4) Discount vs owners
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_discount_percent"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("Discount % vs log(owners_mid) (capped)")
ax.set_xlabel("price_discount_percent")
ax.set_ylabel("log(owners_mid)")
fig.tight_layout()
fig.savefig(VIS_DIR / "discount_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

print("Saved: price_usd_hist.png, owners_mid_log_hist.png, price_vs_log_owners_scatter.png, discount_vs_log_owners_scatter.png")


In [ ]:
# 5) Boxplot: price by top genres (primary_genre)
_top_genres = (
    _df["primary_genre"].value_counts(dropna=True).head(12).index.tolist()
)
_df_gen = _df[_df["primary_genre"].isin(_top_genres)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=_df_gen,
    x="primary_genre",
    y="price_usd",
    ax=ax,
    showfliers=False,
)
ax.set_title("Price distribution by primary_genre (top 12)")
ax.set_xlabel("primary_genre")
ax.set_ylabel("price_usd")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "price_by_primary_genre_box.png", dpi=200)
plt.close(fig)

# 6) Mean log owners by primary genre (top 12)
fig, ax = plt.subplots(figsize=(10, 5))
mean_by_genre = (
    _df_gen.groupby("primary_genre")["log_owners_mid"].mean().sort_values(ascending=False)
)
mean_by_genre.plot(kind="bar", ax=ax)
ax.set_title("Mean log(owners_mid) by primary_genre (top 12)")
ax.set_xlabel("primary_genre")
ax.set_ylabel("mean log(owners_mid)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "mean_log_owners_by_primary_genre_bar.png", dpi=200)
plt.close(fig)

print("Saved: price_by_primary_genre_box.png, mean_log_owners_by_primary_genre_bar.png")


In [ ]:
# 7) Correlation heatmap for numeric features
num_cols = [
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "age_days",
    "steamspy_ccu",
    "steamspy_average_forever",
    "steamspy_average_2weeks",
    "steamspy_median_forever",
    "steamspy_median_2weeks",
    "developer_count",
    "publisher_count",
    "tag_count",
]
num_cols = [c for c in num_cols if c in _df.columns]

corr = _df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    annot=False,
    square=False,
    ax=ax,
)
ax.set_title("Correlation heatmap (numeric features)")
fig.tight_layout()
fig.savefig(VIS_DIR / "correlation_heatmap_numeric.png", dpi=200)
plt.close(fig)

print("Saved: correlation_heatmap_numeric.png")


## 9. Linear regression: optimal price (reference game)

- **Model:** predict `log_owners_mid` from `price_usd` plus non-price controls (`age_days`, tag/developer/publisher counts, top-K `genre__*` flags).
- **Objective:** maximize a **revenue proxy** `price_usd × exp(predicted log_owners_mid)` by sweeping price on a grid while holding a fixed **reference game** profile (median counts, mode genre).
- **Limitation:** cross-sectional Steam catalog data measures historical association with SteamSpy’s owner-range proxy—not causal elasticity or A/B-tested “true” optimal pricing.

In [ ]:
import importlib.util
import subprocess
import sys
import warnings
from pathlib import Path

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

if "VIS_DIR" not in globals():
    VIS_DIR = SCRIPT_DIR / "visualizations"
    VIS_DIR.mkdir(parents=True, exist_ok=True)

genre_cols = [c for c in df_clean.columns if c.startswith("genre__")]
feature_cols = [
    "price_usd",
    "age_days",
    "tag_count",
    "developer_count",
    "publisher_count",
    *genre_cols,
]
model_df = df_clean.dropna(
    subset=["log_owners_mid", "price_usd", *feature_cols[1:]]
).copy()

X = model_df[feature_cols].astype(np.float64)
y = model_df["log_owners_mid"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


def predict_log_owners(model, X_df: pd.DataFrame) -> np.ndarray:
    """Predict without BLAS RuntimeWarning noise from ill-conditioned matmul."""
    X_arr = X_df[feature_cols].astype(np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
            return model.predict(X_arr)


reg = LinearRegression(fit_intercept=True)
reg.fit(X_train, y_train)

y_train_pred = predict_log_owners(reg, X_train)
y_test_pred = predict_log_owners(reg, X_test)
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

price_coef_idx = feature_cols.index("price_usd")
price_coef = reg.coef_[price_coef_idx]

print(f"Rows used: {len(model_df):,}")
print(f"R² (train): {r2_train:.4f}")
print(f"R² (test):  {r2_test:.4f}")
print(f"price_usd coefficient: {price_coef:.6f}")

# Reference game: median counts, genre flags from mode primary_genre
ref = {}
for col in ["age_days", "tag_count", "developer_count", "publisher_count"]:
    ref[col] = float(X_train[col].median())

mode_genre = model_df["primary_genre"].mode(dropna=True).iloc[0]
mode_genre_col = f"genre__{sanitize_feature_name(str(mode_genre))}"
for col in genre_cols:
    ref[col] = 1.0 if col == mode_genre_col else 0.0

ref_row = {**ref, "price_usd": float(X_train["price_usd"].median())}
ref_X = pd.DataFrame([ref_row], columns=feature_cols)

p_lo = float(model_df["price_usd"].quantile(0.01))
p_hi = float(model_df["price_usd"].quantile(0.99))
price_grid = np.linspace(p_lo, p_hi, 200)

ref_base = ref_X.drop(columns=["price_usd"]).iloc[0]
grid_rows = []
for p in price_grid:
    row = ref_base.copy()
    row["price_usd"] = p
    grid_rows.append(row)
grid_X = pd.DataFrame(grid_rows, columns=feature_cols)

pred_log_owners = predict_log_owners(reg, grid_X)
pred_owners = np.exp(pred_log_owners)
revenue = price_grid * pred_owners

p_opt_idx = int(np.argmax(revenue))
p_opt = float(price_grid[p_opt_idx])
rev_opt = float(revenue[p_opt_idx])
owners_opt = float(pred_owners[p_opt_idx])

p_median = float(X_train["price_usd"].median())
ref_median = ref_X.copy()
ref_median["price_usd"] = p_median
log_at_median = float(predict_log_owners(reg, ref_median)[0])
owners_at_median = float(np.exp(log_at_median))
rev_at_median = p_median * owners_at_median

print()
print("Reference profile:")
print(f"  primary_genre (mode): {mode_genre}")
print(f"  age_days (median): {ref['age_days']:.0f}")
print(f"  tag_count (median): {ref['tag_count']:.0f}")
print()
print(f"Price grid: ${p_lo:.2f} – ${p_hi:.2f} (1st–99th percentile of catalog)")
print(f"Optimal price (revenue proxy max): ${p_opt:.2f}")
print(f"  Predicted owners_mid: {owners_opt:,.0f}")
print(f"  Predicted revenue proxy: ${rev_opt:,.0f}")
print()
print(f"At median catalog price (${p_median:.2f}) on same profile:")
print(f"  Predicted owners_mid: {owners_at_median:,.0f}")
print(f"  Predicted revenue proxy: ${rev_at_median:,.0f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(price_grid, revenue, color="steelblue", lw=2)
ax.axvline(p_opt, color="crimson", ls="--", lw=1.5, label=f"optimal ${p_opt:.2f}")
ax.axvline(
    p_median,
    color="gray",
    ls=":",
    lw=1.5,
    label=f"median catalog ${p_median:.2f}",
)
ax.set_title("Predicted revenue proxy vs price (reference game profile)")
ax.set_xlabel("price_usd")
ax.set_ylabel("revenue proxy = price × exp(pred log owners)")
ax.legend()
fig.tight_layout()
out_path = VIS_DIR / "optimal_price_revenue_curve.png"
fig.savefig(out_path, dpi=200)
plt.close(fig)
print("\nSaved to Visualisations.")

### Findings (reference-game linear model)

The cell above fits ordinary least squares on **log(owners_mid)** using store price, game age, tag/developer/publisher counts, and top-K genre flags across the cleaned single-player catalog. **Test R² is modest (on the order of 0.30)**—price and these controls capture some of the spread in estimated owners, but most of what drives success (quality, marketing, timing, franchise effects) is not in the model.

**Price coefficient:** When `price_usd` enters with a **small positive** coefficient, higher listed prices in the data tend to co-occur with *higher* estimated owners, not lower. That pattern is what you expect in a **cross-section** of released games: hits often charge more *and* sell more. It is **not** evidence that raising price causes more sales. Do not treat this slope as causal elasticity or as a pricing recommendation.

**Reference profile:** The optimum is computed for a synthetic “typical” game—mode primary genre (usually **Action**), median `age_days` and tag/developer/publisher counts, with genre dummies set for that mode. It is a baseline scenario, not the optimal price for any one title in the table.

**Revenue proxy** (price × predicted owners on that profile): Compare the printed **grid maximum** vs **median catalog price**. When the predicted revenue curve slopes upward across the whole swept range (1st–99th percentile of catalog prices), the reported “optimal” price lands on the **top of the grid** (e.g. ~$40)—an artifact of the positive price association, not a discovered sweet spot near the median (~$6).

**Chart:** `optimal_price_revenue_curve.png` — blue curve = predicted revenue proxy vs price; red dashed = grid optimum; grey dotted = median catalog price on the same profile.

**Takeaway:** Useful for **exploring associations** in historical Steam + SteamSpy data. For real launch pricing you would need causal or experimental evidence, richer sales data than owner-range midpoints, and a game-specific feature profile—not this catalog-wide linear fit.